#### **00-data**

The [router](https://github.com/atOCEANO/exchange-router-service) serves public
market data from several exchanges behind one schema. No API key, no account.
This notebook covers what you need to read that data correctly, before any
simulation is involved.

How to write a time window, what the HTTP response looks like, what the SDK
turns it into, the places a candle frame can mislead you, and how to pull a
whole venue in one call.

#### **Setup**

The environment is built before this cell runs. `docker compose up` installs
the client and starts a router. A pip install next to a router you started
yourself does the same thing.

So this cell checks and opens the client. It installs nothing and starts
nothing. If the client is missing, or nothing answers on the router address, it
stops here and prints the command to run, rather than failing as an ImportError
two sections down or a connection error that does not say what to start.

A router that answers is not the same as a router that can reach an
exchange. `/status` is served locally and says nothing about the internet
beyond it. If a fetch further down hangs or errors, look there first: the
container needs outbound access, and some networks and jurisdictions block the
exchange endpoints outright.

Two versions print. The service and the [client](https://github.com/atOCEANO/exchange-router-service/blob/main/.Documentation/Python_SDK.md)
are released separately. Every frame also carries a schema number, printed a
few sections down, which is the field that says a given pair agree on the
shape of the data.

The client is opened once and used for the whole notebook.
`with ExchangeRouterClient(base_url=url) as client:` also works, and is the
better shape in a script. In a notebook the block ends with the cell, so every
later cell would have to open its own. `client.close()` at the end does what
leaving the block would do.

In [1]:
import os
import urllib.request

ROUTER_URL = os.environ.get(key="ROUTER_URL", default="http://127.0.0.1:8040")

try:
    import exchange_router_client
except ImportError as error:
    raise ImportError(f"{error.name} is missing; run 'docker compose up' from "
                      f"this repository, or pip install -r requirements.txt") from None


def router_is_up(url):
    try:
        with urllib.request.urlopen(url=f"{url}/status", timeout=2.0) as response:
            return response.status == 200
    except Exception:
        return False


if not router_is_up(url=ROUTER_URL):
    raise RuntimeError(f"no router answering on {ROUTER_URL}; run 'docker compose "
                       f"up' from this repository, or start one yourself and point "
                       f"ROUTER_URL at it")

client = exchange_router_client.ExchangeRouterClient(base_url=ROUTER_URL)

print("service    ", client.get_version(), "at", ROUTER_URL)
print("client     ", exchange_router_client.__version__)
print("exchanges  ", ", ".join(client.get_exchanges()))

service     2.5.6 at http://exchange-router-service:8040
client      5.1.0
exchanges   binance, bybit, hyperliquid, kraken, kucoin, okx


#### **Writing a window**

The router expects times as Unix milliseconds. Unix time is the count of
seconds since 1 January 1970 UTC. Python gives you seconds, the router wants
milliseconds, so multiply by 1000.

**Always pass a timezone.** `datetime(year=2026, month=1, day=1)` carries no
timezone, so Python resolves it against the machine's clock. The same line then
means a different instant on different machines, and nothing raises.

The cell below resolves that one date against four named zones instead of the
local one, so the spread is the same wherever you run it. **Lisbon and UTC
agree in January.** That is why this bug survives: it costs nothing until the
person who wrote the code is not the person running it, or until the clocks
change in March.

Going the other way needs nothing. The frame the client returns is already
indexed by tz-aware UTC timestamps.

In [2]:
import datetime
import zoneinfo


def utc_millis(year, month, day):
    moment = datetime.datetime(year=year, month=month, day=day,
                               tzinfo=datetime.timezone.utc)
    return int(moment.timestamp() * 1000)


def utc_datetime(millis):
    return datetime.datetime.fromtimestamp(timestamp=millis / 1000,
                                           tz=datetime.timezone.utc)


correct = utc_millis(year=2026, month=1, day=1)

#  The same written date, resolved as a naive datetime would be on a machine set
#  to each of these zones.
for name in ("UTC", "Europe/Lisbon", "Asia/Tokyo", "America/New_York"):
    moment = datetime.datetime(year=2026, month=1, day=1,
                               tzinfo=zoneinfo.ZoneInfo(key=name))
    millis = int(moment.timestamp() * 1000)
    print(f"{name:<18} {millis}  {(millis - correct) / 3_600_000:+.0f} hours")

print(f"\nthis kernel        {datetime.datetime.now().astimezone().tzinfo}")

UTC                1767225600000  +0 hours
Europe/Lisbon      1767225600000  +0 hours
Asia/Tokyo         1767193200000  -9 hours
America/New_York   1767243600000  +5 hours

this kernel        UTC


#### **The candles**

**The parameter is called `start`, and it is the end of the window.** The router
walks backwards from it. A `start` of 1 January 2026 with a year of hourly bars
returns 2025, not 2026. Read it as a start date and you get the wrong year with
no error. The router calls
the behaviour an [inclusive backward-walking upper
bound](https://github.com/atOCEANO/exchange-router-service/blob/main/.Documentation/API_Reference.md#pagination-semantics).

`start` is optional. Leave it out and you get the most recent bars, ending
now, which is a different window every time you run it. Pinning it is what makes
a result reproducible without committing a data file.

Each timestamp is the bar's opening time, in UTC. So the row labelled 09:00
covers 09:00 to 10:00, and its close is the price at 10:00. Every adapter the
router carries is normalised to that convention.

The cell prints the dates it actually received, so you never have to trust the
constant.

In [3]:
EXCHANGE = "binance"
MARKET   = "spot"
SYMBOL   = "BTCUSDT"
INTERVAL = "1h"

ANCHOR = utc_millis(year=2026, month=1, day=1)
BARS   = 8760

candles = client.get_candles(
    exchange=EXCHANGE,
    market_type=MARKET,
    symbol=SYMBOL,
    interval=INTERVAL,
    limit=BARS,
    start=ANCHOR,
)

print(f"asked for {BARS} bars ending {utc_datetime(millis=ANCHOR)}")
print(f"got {len(candles)}, {candles.index[0]} to {candles.index[-1]}")
print(f"columns: {', '.join(candles.columns)}")

candles.tail()

asked for 8760 bars ending 2026-01-01 00:00:00+00:00
got 8760, 2025-01-01 01:00:00+00:00 to 2026-01-01 00:00:00+00:00
columns: open, high, low, close, volume, volume_usd


,open,high,low,close,volume,volume_usd
datetime,,,,,,
2025-12-31 20:00:00+00:00,87541.85,87712.13,87250.00,87662.01,770.32743,6.752845e+07
2025-12-31 21:00:00+00:00,87662.01,87879.04,87662.01,87799.97,383.85550,3.370250e+07
2025-12-31 22:00:00+00:00,87799.97,87879.04,87656.19,87728.29,236.18697,2.072028e+07
2025-12-31 23:00:00+00:00,87728.30,87728.30,87619.67,87648.22,191.19123,1.675757e+07
2026-01-01 00:00:00+00:00,87648.21,87849.26,87632.74,87809.23,233.66036,2.051754e+07


#### **What the frame kept**

**The SDK does not hand back the response.** It builds a DataFrame from it, and
the two are not the same shape.

The cell below calls the same route by hand, with the standard library, so you
can see the JSON. **Two things differ from the call above.** It asks for three
bars rather than a year, because the point is the shape of one row. And
`market_type` is a path segment in the URL where the client took it as a keyword
argument.

One `get_candles` is not one HTTP request. No exchange returns a year of
hourly bars at once, so the client asks repeatedly, walking backwards, and
stitches the pages together. What prints below is one row of one page.

Every row repeats what is constant across the request: the symbol, the market
type, the quote asset (the currency a price is denominated in, USDT here) and
the interval. And **volume is an object, not a number.**

In [4]:
import json
import urllib.parse

query = urllib.parse.urlencode(
    query={"interval": INTERVAL, "limit": 3, "start": ANCHOR},
)
route = f"{ROUTER_URL}/{EXCHANGE}/{MARKET}/candles/{SYMBOL}?{query}"

print(route)

with urllib.request.urlopen(url=route, timeout=30.0) as response:
    rows = json.load(fp=response)

print(f"\n{len(rows)} rows, showing the last one\n")
print(json.dumps(obj=rows[-1], indent=2))

http://exchange-router-service:8040/binance/spot/candles/BTCUSDT?interval=1h&limit=3&start=1767225600000



3 rows, showing the last one

{
  "symbol": "BTCUSDT",
  "market_type": "spot",
  "quote": "USDT",
  "interval": "1h",
  "timestamp": 1767225600000,
  "open": 87648.21,
  "high": 87849.26,
  "low": 87632.74,
  "close": 87809.23,
  "volume": {
    "native": 233.66036,
    "unit": "base",
    "contract_size": null,
    "usd": 20517536.2931228,
    "usd_basis": {
      "method": "close",
      "close": null,
      "close_ts": null
    }
  }
}


Constant fields move to `df.attrs`. Per-bar fields stay columns. That is why
the frame is narrow and still knows where it came from. The schema number the
setup section mentioned is in there, alongside the venue, the symbol and the
volume unit.

**attrs are not columns.** Concatenate BTC from two exchanges and the rows will
not say which is which. `with_provenance` copies the exchange, symbol, quote and
unit back into columns for that case. It is a module-level function rather than
a client method, so it is not something you would find by exploring `client`.

In [5]:
for key, value in candles.attrs.items():
    print(f"{key:<16} {value}")

exchange_router_client.with_provenance(df=candles).tail(n=3)

exchange         binance
market_type      spot
symbol           BTCUSDT
schema_version   3
warnings         []
quote            USDT
interval         1h
volume_unit      base
contract_size    None
usd_basis        close


,open,high,low,close,volume,volume_usd,exchange,symbol,quote,unit
datetime,,,,,,,,,,
2025-12-31 22:00:00+00:00,87799.97,87879.04,87656.19,87728.29,236.18697,2.072028e+07,binance,BTCUSDT,USDT,base
2025-12-31 23:00:00+00:00,87728.30,87728.30,87619.67,87648.22,191.19123,1.675757e+07,binance,BTCUSDT,USDT,base
2026-01-01 00:00:00+00:00,87648.21,87849.26,87632.74,87809.23,233.66036,2.051754e+07,binance,BTCUSDT,USDT,base


#### **Volume is two questions**

First, the three market types, because they are what makes volume ambiguous.
**Spot** is the coin itself, bought outright. A **linear** perpetual is a
contract priced in the quote currency, USDT here, that never expires. An
**inverse** perpetual is priced in the coin instead: one contract is worth a
fixed number of dollars, and the coin is what you post and get paid in.

So a volume figure raises two questions, and the column name answers neither.

**What is one unit?** `attrs["volume_unit"]` says. Base coin on spot and linear,
contracts on inverse. The same column under the same name is counting different
things, so adding the two together is meaningless.

**What is it worth?** `attrs["usd_basis"]` says how volume_usd was reached:
multiplied by the close where volume is coins, multiplied by
`attrs["contract_size"]` where it is contracts, that being the dollars one
contract represents. A converted number without its basis cannot be compared
across venues.

The inverse symbol below is looked up rather than typed, because the naming
differs per exchange. It takes the first BTC market the venue lists, and the
output names which one that turned out to be.

In [6]:
def describe_volume(frame):
    attrs = frame.attrs
    return (f"{attrs['market_type']:<8} {attrs['symbol']:<14} "
            f"unit={attrs['volume_unit']:<9} basis={attrs['usd_basis']:<14} "
            f"contract_size={attrs['contract_size']}")


markets = client.get_markets(exchange=EXCHANGE, market_type="inverse")
inverse = [market["symbol"] for market in markets["markets"]
           if market["symbol"].startswith("BTC")]

contracts = client.get_candles(
    exchange=EXCHANGE,
    market_type="inverse",
    symbol=inverse[0],
    interval=INTERVAL,
    limit=3,
)

print(f"inverse BTC markets: {len(inverse)}, using {inverse[0]}\n")
print(describe_volume(frame=candles))
print(describe_volume(frame=contracts))

inverse BTC markets: 1, using BTCUSD

spot     BTCUSDT        unit=base      basis=close          contract_size=None
inverse  BTCUSD         unit=contract  basis=contract_size  contract_size=100.0


#### **When it comes back short**

Ask for more history than a symbol has and you get fewer bars, with nothing in
the frame to say so.

**The client warns.** It raises a `RouterDataWarning` whenever a route that
pages, which candles do, comes back shorter than the limit asked for. This is
**on by default**: the client's constructor takes `verbose=True`, and every call
inherits it unless you pass your own.

Two limits. It is a warning and not an exception, so **nothing stops** and the
short frame carries on into whatever you do next. And Python prints a given
warning once per place it is raised, so the second short result in a session is
silent. `get_candles(..., verbose=False)`, or `client.verbose = False` for all
of them, turns it off.

That is why the candles cell prints what it asked for next to what it got. The
cell below catches the warning instead of printing it, and `simplefilter`
defeats the once-per-place rule.

In [7]:
import warnings

#  BTCUSDT was listed in August 2017, so a year of history ending here cannot
#  exist. If this ever stops coming back short, the date needs moving earlier.
EARLY = utc_millis(year=2017, month=9, day=1)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter(action="always")

    early = client.get_candles(
        exchange=EXCHANGE,
        market_type=MARKET,
        symbol=SYMBOL,
        interval=INTERVAL,
        limit=BARS,
        start=EARLY,
    )

print(f"asked for {BARS} bars ending {utc_datetime(millis=EARLY)}")
print(f"got {len(early)}, {early.index[0]} to {early.index[-1]}")

for entry in caught:
    print(f"\n{entry.category.__name__}: {entry.message}")

asked for 8760 bars ending 2017-09-01 00:00:00+00:00
got 357, 2017-08-17 04:00:00+00:00 to 2017-09-01 00:00:00+00:00



#### **The whole venue**

`candles_many` takes a list of symbols and returns a BatchResult. **Three
buckets, not one answer**: clean, degraded with a warning saying why, and
failed with the exception that caused it. One delisted pair does not cost you
the rest.

`report()` prints all three. Everything else on the result, including `len` and
indexing by symbol, walks clean plus degraded, so **a symbol that failed is
absent rather than empty** and the length is not the number you requested.

Many come back degraded. A binance spot symbol is base then quote.
BTCUSDT is bitcoin priced in USDT, but ETHBTC is ether priced in
**bitcoin**, and BNBETH is BNB priced in **ether**. The router still fills
volume_usd for those, computed off the close in whatever the quote asset
happens to be, so the figure is denominated in BTC or ETH and its name lies.
The warning names the asset per symbol:

```
ETCBTC   quote='BTC'; *_usd fields are quote-denominated (BTC), not US dollars
ZECETH   quote='ETH'; *_usd fields are quote-denominated (ETH), not US dollars
```

**Summing volume_usd across a mixed batch gives a number with no unit.**
`attrs["quote"]` is the field to filter on.

The batch summary warning is attributed to a file inside asyncio, because the
client raises it from the thread its event loop runs on. It is about your call,
not about asyncio.

You do not manage rate limits. The router does. The binance adapter reads
the exchange's weight header on every response and pauses that host at 95% of
the limit. A 429 sets a backoff from the Retry-After header. That state is held
per host inside the router, so every caller shares one budget. `candles_many`
also takes `max_concurrent`, which caps how many requests are in flight.

The list is sliced so the cell finishes quickly. **Remove the slice and the same
call pulls the whole venue.** The printed symbol count is the multiplier on how
long that takes.

In [8]:
import collections

HOW_MANY = 40

markets = client.get_markets(exchange=EXCHANGE, market_type=MARKET)
symbols = [market["symbol"] for market in markets["markets"]]

pulled = client.candles_many(
    exchange=EXCHANGE,
    market_type=MARKET,
    symbols=symbols[:HOW_MANY],
    interval="1d",
    limit=30,
)

print(f"{EXCHANGE} {MARKET} lists {len(symbols)} symbols, pulled {HOW_MANY}")
print(pulled.report())

quotes = collections.Counter(frame.attrs["quote"] for _, frame in pulled.items())
print(f"\nquote assets returned: {dict(quotes)}")

name, frame = next(pulled.items())
print(f"\n{name}")
frame.tail(n=3)

binance spot lists 1362 symbols, pulled 40
40 requested: 20 ok, 20 degraded, 0 failed
  degraded
    ETHBTC   candles binance/spot/ETHBTC: quote='BTC'; *_usd fields are quote-denominated (BTC), not US dollars
    LTCBTC   candles binance/spot/LTCBTC: quote='BTC'; *_usd fields are quote-denominated (BTC), not US dollars
    BNBBTC   candles binance/spot/BNBBTC: quote='BTC'; *_usd fields are quote-denominated (BTC), not US dollars
    BNBETH   candles binance/spot/BNBETH: quote='ETH'; *_usd fields are quote-denominated (ETH), not US dollars
    LINKBTC   candles binance/spot/LINKBTC: quote='BTC'; *_usd fields are quote-denominated (BTC), not US dollars
    LINKETH   candles binance/spot/LINKETH: quote='ETH'; *_usd fields are quote-denominated (ETH), not US dollars
    ETCBTC   candles binance/spot/ETCBTC: quote='BTC'; *_usd fields are quote-denominated (BTC), not US dollars
    ZECBTC   candles binance/spot/ZECBTC: quote='BTC'; *_usd fields are quote-denominated (BTC), not US dollars
   

/usr/local/lib/python3.11/asyncio/events.py:84: RouterDataWarning: candles_many binance/spot: 40 requested: 20 ok, 20 degraded, 0 failed. See .report().
  self._context.run(self._callback, *self._args)


,open,high,low,close,volume,volume_usd
datetime,,,,,,
2026-09-05 00:00:00+00:00,79660.77,80200.00,79442.0,79831.75,9117.24020,7.278452e+08
2026-09-06 00:00:00+00:00,79831.75,80559.99,79233.0,80341.83,8854.81827,7.114123e+08
2026-09-07 00:00:00+00:00,80341.83,80443.99,80061.9,80151.12,344.73280,2.763072e+07


#### **Closing**

`candles` is ready to backtest as it stands.
[emsl](https://github.com/atOCEANO/embeddable-market-simulation-library) reads
the open, high, low, close and volume columns directly and never uses the index
to line anything up, so the bar order you have here is the order it simulates.
It does read the index for one thing, how many bars a year holds, which is what
annualizes a Sharpe correctly on hourly data. It does not read attrs, which is
why the volume unit is something you have to know rather than something that
travels with the numbers.

The routes this notebook skipped behave the same way. Trades, aggregated
trades, funding rates, open interest, liquidations and the long-short ratio all
come back through the same frame builder, so everything above about columns,
attrs, units and short results holds for them unchanged.

Two things it does not touch at all. `get_capabilities` reports what a
venue actually supports before you call it, which is how the client refuses an
interval a market does not offer and names the valid ones instead of letting
the exchange answer with an error. And the router streams over a WebSocket as
well as answering over REST, which is a different shape of program from
anything here.

`client.close()` shuts down the background thread the client runs its async
loop on, and its connection pool. It is what leaving a `with` block does.

In [9]:
client.close()